In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# 1. Load Data
CSV_PATH = '/content/drive/MyDrive/CSC 201/outbreaks.csv'
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)

    # 2. Text & Label Preparation
    def make_text(row):
        return (f"A foodborne outbreak occurred in {row['State']} during {row['Year']}. "
                f"The pathogen was {row['Species']}. The suspected food source was {row['Food']}. "
                f"There were {row['Illnesses']} illnesses, {row['Hospitalizations']} hospitalizations, and {row['Fatalities']} deaths.")

    df['text'] = df.apply(make_text, axis=1)
    df['label'] = ((df['Hospitalizations'] >= 10) | (df['Fatalities'] > 0) | (df['Illnesses'] > 10)).astype(int)

    display(df.head())
else:
    print(f"Error: File not found at {CSV_PATH}. Please check your Drive connection.")

,Year,Month,State,Location,Food,Ingredient,Species,Serotype/Genotype,Status,Illnesses,Hospitalizations,Fatalities,text,label
0,1998,January,California,Restaurant,NaN,NaN,NaN,NaN,NaN,20,0.0,0.0,A foodborne outbreak occurred in California du...,1
1,1998,January,California,NaN,Custard,NaN,NaN,NaN,NaN,112,0.0,0.0,A foodborne outbreak occurred in California du...,1
2,1998,January,California,Restaurant,NaN,NaN,NaN,NaN,NaN,35,0.0,0.0,A foodborne outbreak occurred in California du...,1
3,1998,January,California,Restaurant,"Fish, Ahi",NaN,Scombroid toxin,NaN,Confirmed,4,0.0,0.0,A foodborne outbreak occurred in California du...,0
4,1998,January,California,Private Home/Residence,"Lasagna, Unspecified; Eggs, Other",NaN,Salmonella enterica,Enteritidis,Confirmed,26,3.0,0.0,A foodborne outbreak occurred in California du...,1


In [ ]:
def make_text(row):
    return (
        f"A foodborne outbreak occurred in {row['State']} "
        f"during {row['Year']}. "
        f"The pathogen was {row['Species']}. "
        f"The suspected food source was {row['Food']}. "
        f"There were {row['Illnesses']} illnesses, "
        f"{row['Hospitalizations']} hospitalizations, "
        f"and {row['Fatalities']} deaths."
    )

df["text"] = df.apply(make_text, axis=1)

In [ ]:
def make_text(row):
    return (
        f"A foodborne outbreak occurred in {row['State']} "
        f"during {row['Year']}. "
        f"The pathogen was {row['Species']}. "
        f"The suspected food source was {row['Food']}. "
        f"There were {row['Illnesses']} illnesses, "
        f"{row['Hospitalizations']} hospitalizations, "
        f"and {row['Fatalities']} deaths."
    )

df["text"] = df.apply(make_text, axis=1)

In [ ]:
df["label"] = (
    (df["Hospitalizations"] >= 10) |
    (df["Fatalities"] > 0) |
    (df["Illnesses"] > 10)
).astype(int)

In [ ]:
!pip install transformers datasets torch accelerate xgboost

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Re-initializing BERT model after renaming the XGBoost instance
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer

# Convert pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df, preserve_index=False)

Now, let's tokenize the `text` column in the dataset. This prepares the text data for input into the BERT model.

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/19119 [00:00<?, ? examples/s]

Next, we'll split the tokenized dataset into training and testing sets. This is crucial for evaluating the model's performance on unseen data. We'll use 80% for training and 20% for testing.

In [ ]:
train_test_split = tokenized_datasets.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

Now, we will define the `TrainingArguments` and initialize the `Trainer` for fine-tuning PubMedBERT on our severity prediction task. We'll specify output directory, learning rate, batch size, and number of epochs.

In [ ]:
# 3. Fine-tune PubMedBERT
MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

hf_dataset = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)
split = tokenized_datasets.train_test_split(test_size=0.2)

training_args = TrainingArguments(
    output_dir='./fine_tuned_model',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    optim='adamw_torch', # Explicitly use non-fused optimizer to avoid XLA errors
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test']
)

trainer.train()
trainer.save_model('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

Map:   0%|          | 0/19119 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.044237,0.000029
2,0.001818,0.000019
3,0.001998,0.000013


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/tokenizer.json')

### Test the Fine-tuned Model
Use the function below to provide a custom outbreak description and see the model's prediction.

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Point to our specific saved model directory
MODEL_PATH = './fine_tuned_model'
BASE_MODEL = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if os.path.exists(MODEL_PATH):
    print(f"Loading your fine-tuned model from {MODEL_PATH}...")
    test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    test_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
else:
    print("Warning: Fine-tuned model folder not found. Falling back to base model.")
    test_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    test_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2).to(device)

def predict_severity(text):
    test_model.eval()
    inputs = test_tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=256).to(device)

    with torch.no_grad():
        logits = test_model(**inputs).logits

    probabilities = torch.softmax(logits, dim=1)
    prediction = torch.argmax(probabilities, dim=1).item()
    conf = probabilities[0][prediction].item()

    severity = 'Severe (High impact)' if prediction == 1 else 'Not Severe (Low impact)'
    print(f'\nInput: {text}')
    print(f'Result: {severity} (Confidence: {conf:.2%})')

# Test scenarios
scenarios = [
    'Large Salmonella outbreak at a wedding. 40 people sick, 12 in the hospital.',
    'Suspected E. coli in spinach. Only 2 people reported mild stomach cramps.',
    'Deadly Listeria outbreak linked to cantaloupe. 15 cases and 3 fatalities.'
]

for s in scenarios:
    predict_severity(s)

Loading your fine-tuned model from ./fine_tuned_model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Input: Large Salmonella outbreak at a wedding. 40 people sick, 12 in the hospital.
Result: Severe (High impact) (Confidence: 100.00%)

Input: Suspected E. coli in spinach. Only 2 people reported mild stomach cramps.
Result: Not Severe (Low impact) (Confidence: 99.74%)

Input: Deadly Listeria outbreak linked to cantaloupe. 15 cases and 3 fatalities.
Result: Severe (High impact) (Confidence: 100.00%)


After fine-tuning, we can evaluate the model on the test set.

In [ ]:
import torch
import pandas as pd
import numpy as np

# Create state_monthly aggregation needed for merging
state_monthly = (
    df.groupby(["Year", "Month", "State"])
      .size()
      .reset_index(name="outbreak_count")
)

# Function to extract BERT embeddings from the fine-tuned model
def get_bert_embeddings(text_list):
    model.eval()
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        # Using the base model (bert) hidden states before the classification head
        outputs = model.bert(**inputs)
    # Mean pooling of the last hidden state to get a single vector per outbreak description
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Extracting embeddings for a sample (processing full dataset might be slow)
print("Extracting BERT embeddings...")
embeddings = get_bert_embeddings(df['text'].iloc[:1000].tolist())
df_embeddings = pd.DataFrame(embeddings, index=df.index[:1000])
df_embeddings[['Year', 'Month', 'State']] = df[['Year', 'Month', 'State']].iloc[:1000]

# Aggregate embeddings by Year, Month, State to align with counts
agg_embeddings = df_embeddings.groupby(['Year', 'Month', 'State']).mean().reset_index()

# Merge embeddings with the aggregated counts
final_forecasting_data = pd.merge(state_monthly, agg_embeddings, on=['Year', 'Month', 'State'])
print(f"Combined data shape: {final_forecasting_data.shape}")
display(final_forecasting_data.head())

Extracting BERT embeddings...
Combined data shape: (236, 772)


,Year,Month,State,outbreak_count,0,1,2,3,4,5,...,758,759,760,761,762,763,764,765,766,767
0,1998,April,California,8,0.032338,0.010449,0.397760,0.115015,-0.208838,-0.118623,...,0.094171,-0.149592,0.158392,0.389465,-0.061466,-0.137703,-0.216862,-0.075573,0.341155,-0.102232
1,1998,April,Colorado,3,0.032203,0.064324,0.446005,0.126569,-0.251072,-0.133690,...,0.107406,-0.079987,0.195849,0.452502,-0.077639,-0.084802,-0.221556,-0.059879,0.429744,-0.070306
2,1998,April,Florida,25,-0.104913,-0.072816,-0.066467,-0.232300,-0.526481,0.075358,...,-0.107851,0.289281,-0.108830,0.782202,0.077600,-0.396096,-0.103368,0.297537,0.345299,0.158440
3,1998,April,Georgia,1,0.167926,0.131612,0.645794,0.304552,-0.006878,-0.315050,...,0.304420,-0.288846,0.322254,0.181845,-0.335622,-0.006297,-0.248117,-0.200849,0.415680,-0.151972
4,1998,April,Hawaii,2,-0.032120,0.017921,0.224933,-0.054017,-0.384622,-0.037824,...,-0.053963,0.012614,0.028551,0.621419,-0.010197,-0.161005,-0.174590,0.002035,0.395809,-0.018855


In [ ]:
# 4. Extract Embeddings and Train Forecaster
import torch
import numpy as np

# Specific XLA handling for Colab TPU
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print(f"Using XLA device: {device}")
except ImportError:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"XLA not found, using: {device}")

def get_embeddings(text_list, batch_size=32):
    model.to(device)
    model.eval()
    embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        # Move inputs explicitly to the XLA device
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.bert(**inputs)
        # Move results back to CPU for storage
        embeddings.append(outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy())
    return np.vstack(embeddings)

# Create Features from Embeddings
print("Extracting embeddings for forecasting (this may take a minute)... ")
embeddings_array = get_embeddings(df['text'].tolist())
emb_cols = [f'bert_{i}' for i in range(embeddings_array.shape[1])]
df_emb = pd.DataFrame(embeddings_array, columns=emb_cols)
df_feat = pd.concat([df[['Year', 'Month', 'State']], df_emb], axis=1)

# Aggregate and Merge
month_map = {m: i+1 for i, m in enumerate(['January','February','March','April','May','June','July','August','September','October','November','December'])}
agg_data = df_feat.groupby(['Year', 'Month', 'State']).mean().reset_index()
counts = df.groupby(['Year', 'Month', 'State']).size().reset_index(name='outbreak_count')
final_df = pd.merge(counts, agg_data, on=['Year', 'Month', 'State'])

# Trigonometric Time Features
final_df['m_num'] = final_df['Month'].map(month_map)
final_df['sin_m'] = np.sin(2 * np.pi * final_df['m_num'] / 12)
final_df['cos_m'] = np.cos(2 * np.pi * final_df['m_num'] / 12)

# Train XGBoost Forecaster
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X = final_df[['sin_m', 'cos_m'] + emb_cols]
y = final_df['outbreak_count']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

forecaster = XGBRegressor(n_estimators=100, learning_rate=0.05)
forecaster.fit(X_train, y_train)
print(f"Forecasting RMSE: {np.sqrt(mean_squared_error(y_test, forecaster.predict(X_test))):.4f}")

XLA not found, using: cuda
Extracting embeddings for forecasting (this may take a minute)... 
Forecasting RMSE: 1.8166


In [ ]:
import torch
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# 1. Extract embeddings for the entire dataset
def get_all_embeddings(text_list, batch_size=32):
    model.eval()
    all_embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(model.device)
        with torch.no_grad():
            outputs = model.bert(**inputs)
        # Pool hidden states
        embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        all_embeddings.append(embeddings)
    return np.vstack(all_embeddings)

print("Extracting all BERT embeddings...")
embeddings = get_all_embeddings(df['text'].tolist())
emb_cols = [f'bert_{i}' for i in range(embeddings.shape[1])]
df_emb = pd.DataFrame(embeddings, columns=emb_cols)
df_combined = pd.concat([df[['Year', 'Month', 'State']], df_emb], axis=1)

# 2. Aggregate BERT features by Year/Month/State
agg_bert = df_combined.groupby(['Year', 'Month', 'State']).mean().reset_index()

# 3. Merge with our count features
final_df = pd.merge(state_monthly, agg_bert, on=['Year', 'Month', 'State'])

# 4. Train final XGBoost model with BERT features
bert_features = ['month_sin', 'month_cos'] + emb_cols
X_bert = final_df[bert_features]
y_bert = final_df['outbreak_count']

xgb_bert = XGBRegressor()
xgb_bert.fit(X_bert, y_bert)

print("XGBoost model with BERT embeddings trained successfully.")

Extracting all BERT embeddings...


KeyError: "['month_sin', 'month_cos'] not in index"

In [ ]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd

# 1. Ensure GPU usage
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# 2. Optimized extraction function with DataLoader
def get_fast_embeddings(text_list, batch_size=128):
    all_embeddings = []
    print(f"Using device: {device}")

    loader = DataLoader(text_list, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch in loader:
            inputs = tokenizer(
                batch,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=128
            ).to(device)

            outputs = model.bert(**inputs)
            # Mean pool and move to CPU immediately to save GPU memory
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

# 3. Run optimized extraction
print("Starting optimized embedding extraction...")
embeddings = get_fast_embeddings(df['text'].tolist())

# 4. Feature Engineering and Merging
emb_cols = [f'bert_{i}' for i in range(embeddings.shape[1])]
df_emb = pd.DataFrame(embeddings, columns=emb_cols, index=df.index)
df_combined = pd.concat([df[['Year', 'Month', 'State']], df_emb], axis=1)

# Aggregate BERT features by month/state
agg_bert = df_combined.groupby(['Year', 'Month', 'State']).mean().reset_index()

# Create the count target and trigonometric time features
state_monthly_local = df.groupby(['Year', 'Month', 'State']).size().reset_index(name='outbreak_count')
month_to_num = {'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6, 'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12}
state_monthly_local['Month_num'] = state_monthly_local['Month'].map(month_to_num)
state_monthly_local['month_sin'] = np.sin(2 * np.pi * state_monthly_local['Month_num'] / 12)
state_monthly_local['month_cos'] = np.cos(2 * np.pi * state_monthly_local['Month_num'] / 12)

# Final Merge
final_df = pd.merge(state_monthly_local, agg_bert, on=['Year', 'Month', 'State'])

print(f"Successfully combined features. Total rows: {len(final_df)}")
display(final_df.head())

In [ ]:
# Aggregating by Year, Month, and State
state_monthly = (
    df.groupby(["Year", "Month", "State"])
      .size()
      .reset_index(name="outbreak_count")
)

# Map Month to numbers for trigonometric features
month_to_num = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}
state_monthly["Month_num"] = state_monthly["Month"].map(month_to_num)
state_monthly["month_sin"] = np.sin(2 * np.pi * state_monthly["Month_num"] / 12)
state_monthly["month_cos"] = np.cos(2 * np.pi * state_monthly["Month_num"] / 12)

In [ ]:
state_monthly["lag1"] = state_monthly["outbreak_count"].shift(1)
state_monthly["lag2"] = state_monthly["outbreak_count"].shift(2)
state_monthly["lag3"] = state_monthly["outbreak_count"].shift(3)

In [ ]:
month_to_num = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}
monthly["Month_num"] = monthly["Month"].map(month_to_num)

monthly["month_sin"] = np.sin(
    2 * np.pi * monthly["Month_num"] / 12
)

monthly["month_cos"] = np.cos(
    2 * np.pi * monthly["Month_num"] / 12
)

In [ ]:
pip install xgboost

In [ ]:
from xgboost import XGBRegressor

features = [
    "lag1",
    "lag2",
    "lag3",
    "month_sin",
    "month_cos"
]

X_simple = monthly[features]
y_simple = monthly["outbreak_count"]

# Renaming to avoid conflict with PubMedBERT 'model'
xgb_model = XGBRegressor()
xgb_model.fit(X_simple, y_simple)

### 💬 Chat with PubMedBERT
Use the cell below to interact with your fine-tuned model. Enter a description of a foodborne illness scenario, and the model will classify its severity.


In [ ]:
def chat_with_bert():
    print("--- PubMedBERT Severity Classifier ---")
    print("Type 'quit' or 'exit' to stop the chat.\n")

    while True:
        user_input = input("Describe a foodborne outbreak: ")

        if user_input.lower() in ['quit', 'exit']:
            print("Ending chat session. Goodbye!")
            break

        if not user_input.strip():
            continue

        # Use the prediction function defined earlier
        predict_severity(user_input)
        print("-" * 30)

# Start the chat
chat_with_bert()